#Limpeza e Padronização dos Dados

In [0]:
# Define o catálogo e os schemas utilizados no pipeline

catalogo = "workspace"

schema_origem = "bronze"

schema_destino = "silver"

print("Origem:", f"{catalogo}.{schema_origem}")

print("Destino:", f"{catalogo}.{schema_destino}")

Origem: workspace.bronze
Destino: workspace.silver


In [0]:
from pyspark.sql import functions as F

# Carrega os pedidos originais da camada Bronze
df_orders = spark.table(
    f"{catalogo}.{schema_origem}.orders"
)

# Exibe a estrutura original dos dados
df_orders.printSchema()

# Exibe uma amostra dos registros
display(df_orders.limit(10))

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- _arquivo_origem: string (nullable = true)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_data_ingestao,_arquivo_origem
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,2026-09-22T23:37:27.669Z,olist_orders_dataset.csv
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,2026-09-22T23:37:27.669Z,olist_orders_dataset.csv
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,2026-09-22T23:37:27.669Z,olist_orders_dataset.csv
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,2026-09-22T23:37:27.669Z,olist_orders_dataset.csv
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,2026-09-22T23:37:27.669Z,olist_orders_dataset.csv
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00,2026-09-22T23:37:27.669Z,olist_orders_dataset.csv
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,null,null,2017-05-09 00:00:00,2026-09-22T23:37:27.669Z,olist_orders_dataset.csv
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00,2026-09-22T23:37:27.669Z,olist_orders_dataset.csv
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00,2026-09-22T23:37:27.669Z,olist_orders_dataset.csv
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00,2026-09-22T23:37:27.669Z,olist_orders_dataset.csv


In [0]:
# Converte as datas e padroniza o status dos pedidos

df_orders_silver = (
    df_orders

    # Padroniza o status dos pedidos
    .withColumn(
        "order_status",
        F.lower(F.trim(F.col("order_status")))
    )

    # Converte as datas para timestamp
    .withColumn(
        "order_purchase_timestamp",
        F.to_timestamp("order_purchase_timestamp")
    )

    .withColumn(
        "order_approved_at",
        F.to_timestamp("order_approved_at")
    )

    .withColumn(
        "order_delivered_carrier_date",
        F.to_timestamp("order_delivered_carrier_date")
    )

    .withColumn(
        "order_delivered_customer_date",
        F.to_timestamp("order_delivered_customer_date")
    )

    .withColumn(
        "order_estimated_delivery_date",
        F.to_timestamp("order_estimated_delivery_date")
    )
)

In [0]:
# Cria indicadores logísticos para análises posteriores

df_orders_silver = (
    df_orders_silver

    # Identifica inconsistências entre status e data de entrega
    .withColumn(
        "status_entrega_consistente",

        F.when(
            (
                (F.col("order_status") == "delivered") &
                F.col("order_delivered_customer_date").isNull()
            )
            |
            (
                (F.col("order_status") != "delivered") &
                F.col("order_delivered_customer_date").isNotNull()
            ),
            False
        ).otherwise(True)
    )

    # Calcula o tempo de entrega em dias
    .withColumn(
        "dias_entrega",

        F.when(
            F.col("order_delivered_customer_date").isNotNull(),

            F.datediff(
                F.col("order_delivered_customer_date"),
                F.col("order_purchase_timestamp")
            )
        )
    )

    # Identifica entregas realizadas após a data prevista
    .withColumn(
        "entrega_atrasada",

        F.when(
            (F.col("order_status") == "delivered") &
            F.col("order_delivered_customer_date").isNotNull() &
            F.col("order_estimated_delivery_date").isNotNull(),

            F.col("order_delivered_customer_date") >
            F.col("order_estimated_delivery_date")
        )
    )
)

In [0]:
# Salva os pedidos tratados na camada Silver

(
    df_orders_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.orders"
    )
)

print("Tabela Silver criada com sucesso!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.orders"
    ).count()
)

Tabela Silver criada com sucesso!
Quantidade de registros: 99441


In [0]:
# Carrega a tabela persistida na camada Silver

df_validacao = spark.table(
    f"{catalogo}.{schema_destino}.orders"
)

# Verifica a quantidade de pedidos e os indicadores criados

display(
    df_validacao.agg(

        F.count("*").alias("total_pedidos"),

        F.sum(
            F.when(
                F.col("status_entrega_consistente") == False,
                1
            ).otherwise(0)
        ).alias("pedidos_inconsistentes"),

        F.sum(
            F.when(
                F.col("entrega_atrasada") == True,
                1
            ).otherwise(0)
        ).alias("entregas_atrasadas"),

        F.sum(
            F.when(
                F.col("entrega_atrasada") == False,
                1
            ).otherwise(0)
        ).alias("entregas_no_prazo"),

        F.sum(
            F.when(
                F.col("entrega_atrasada").isNull(),
                1
            ).otherwise(0)
        ).alias("atraso_nao_determinado")

    )
)

total_pedidos,pedidos_inconsistentes,entregas_atrasadas,entregas_no_prazo,atraso_nao_determinado
99441,14,7826,88644,2971


In [0]:
# Carrega os itens originais da camada Bronze
df_items = spark.table(
    f"{catalogo}.{schema_origem}.order_items"
)

# Padroniza os tipos de dados dos itens dos pedidos
df_items_silver = (
    df_items

    # Converte o identificador sequencial do item para inteiro
    .withColumn(
        "order_item_id",
        F.col("order_item_id").cast("int")
    )

    # Converte os valores financeiros para decimal
    .withColumn(
        "price",
        F.col("price").cast("decimal(18,2)")
    )

    .withColumn(
        "freight_value",
        F.col("freight_value").cast("decimal(18,2)")
    )

    # Converte a data-limite de envio para timestamp
    .withColumn(
        "shipping_limit_date",
        F.to_timestamp("shipping_limit_date")
    )

    # Calcula o valor total do item, incluindo frete
    .withColumn(
        "valor_total_item",
        F.col("price") + F.col("freight_value")
    )
)

# Exibe uma amostra dos dados tratados
display(df_items_silver.limit(10))

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,_data_ingestao,_arquivo_origem,valor_total_item
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.90,13.29,2026-09-22T23:37:08.794Z,olist_order_items_dataset.csv,72.19
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.90,19.93,2026-09-22T23:37:08.794Z,olist_order_items_dataset.csv,259.83
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.00,17.87,2026-09-22T23:37:08.794Z,olist_order_items_dataset.csv,216.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,2026-09-22T23:37:08.794Z,olist_order_items_dataset.csv,25.78
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.90,18.14,2026-09-22T23:37:08.794Z,olist_order_items_dataset.csv,218.04
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23T03:55:27.000Z,21.90,12.69,2026-09-22T23:37:08.794Z,olist_order_items_dataset.csv,34.59
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14T12:10:31.000Z,19.90,11.85,2026-09-22T23:37:08.794Z,olist_order_items_dataset.csv,31.75
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10T12:30:45.000Z,810.00,70.75,2026-09-22T23:37:08.794Z,olist_order_items_dataset.csv,880.75
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26T18:31:29.000Z,145.95,11.65,2026-09-22T23:37:08.794Z,olist_order_items_dataset.csv,157.60
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56.000Z,53.99,11.40,2026-09-22T23:37:08.794Z,olist_order_items_dataset.csv,65.39


In [0]:
# Salva os itens tratados na camada Silver
(
    df_items_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.order_items"
    )
)

print("Tabela order_items criada na Silver!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.order_items"
    ).count()
)

Tabela order_items criada na Silver!
Quantidade de registros: 112650


In [0]:
# Carrega a tabela Silver persistida
df_items_validacao = spark.table(
    f"{catalogo}.{schema_destino}.order_items"
)

# Valida a quantidade de registros e os valores financeiros
display(
    df_items_validacao.agg(

        F.count("*").alias("total_itens"),

        F.sum("price").alias("valor_total_produtos"),

        F.sum("freight_value").alias("valor_total_frete"),

        F.sum("valor_total_item").alias("valor_total_geral"),

        F.min("price").alias("menor_preco"),

        F.max("price").alias("maior_preco")

    )
)

total_itens,valor_total_produtos,valor_total_frete,valor_total_geral,menor_preco,maior_preco
112650,13591643.70,2251909.54,15843553.24,0.85,6735.00


In [0]:
# Carrega os clientes originais da camada Bronze
df_customers = spark.table(
    f"{catalogo}.{schema_origem}.customers"
)

# Padroniza os dados geográficos dos clientes
df_customers_silver = (
    df_customers

    # Converte o prefixo do CEP para texto de cinco caracteres
    .withColumn(
        "customer_zip_code_prefix",
        F.lpad(
            F.col("customer_zip_code_prefix"),
            5,
            "0"
        )
    )

    # Padroniza o nome da cidade
    .withColumn(
        "customer_city",
        F.lower(F.trim(F.col("customer_city")))
    )

    # Padroniza a sigla do estado
    .withColumn(
        "customer_state",
        F.upper(F.trim(F.col("customer_state")))
    )
)

# Exibe uma amostra dos clientes tratados
display(df_customers_silver.limit(10))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,_data_ingestao,_arquivo_origem
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,2026-09-22T23:36:49.287Z,olist_customers_dataset.csv
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP,2026-09-22T23:36:49.287Z,olist_customers_dataset.csv
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP,2026-09-22T23:36:49.287Z,olist_customers_dataset.csv
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP,2026-09-22T23:36:49.287Z,olist_customers_dataset.csv
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,2026-09-22T23:36:49.287Z,olist_customers_dataset.csv
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC,2026-09-22T23:36:49.287Z,olist_customers_dataset.csv
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,04534,sao paulo,SP,2026-09-22T23:36:49.287Z,olist_customers_dataset.csv
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG,2026-09-22T23:36:49.287Z,olist_customers_dataset.csv
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR,2026-09-22T23:36:49.287Z,olist_customers_dataset.csv
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG,2026-09-22T23:36:49.287Z,olist_customers_dataset.csv


In [0]:
# Salva os clientes tratados na camada Silver
(
    df_customers_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.customers"
    )
)

print("Tabela customers criada na Silver!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.customers"
    ).count()
)

Tabela customers criada na Silver!
Quantidade de registros: 99441


In [0]:
# Carrega os clientes tratados
df_customers_validacao = spark.table(
    f"{catalogo}.{schema_destino}.customers"
)

# Verifica a quantidade de clientes e a cobertura geográfica
display(
    df_customers_validacao.agg(

        F.count("*").alias("total_registros"),

        F.countDistinct(
            "customer_unique_id"
        ).alias("clientes_unicos"),

        F.countDistinct(
            "customer_state"
        ).alias("estados_distintos"),

        F.countDistinct(
            "customer_city"
        ).alias("cidades_distintas"),

        F.sum(
            F.when(
                F.col("customer_state").isNull(),
                1
            ).otherwise(0)
        ).alias("estados_nulos")

    )
)

total_registros,clientes_unicos,estados_distintos,cidades_distintas,estados_nulos
99441,96096,27,4119,0


In [0]:
# Carrega os produtos originais da camada Bronze
df_products = spark.table(
    f"{catalogo}.{schema_origem}.products"
)

# Padroniza categorias e converte atributos numéricos
df_products_silver = (
    df_products

    # Trata categorias ausentes
    .withColumn(
        "product_category_name",

        F.when(
            F.col("product_category_name").isNull() |
            (F.trim(F.col("product_category_name")) == ""),

            "nao_informada"
        ).otherwise(
            F.lower(F.trim(F.col("product_category_name")))
        )
    )

    # Converte atributos descritivos para inteiro
    .withColumn(
        "product_name_lenght",
        F.col("product_name_lenght").cast("int")
    )

    .withColumn(
        "product_description_lenght",
        F.col("product_description_lenght").cast("int")
    )

    .withColumn(
        "product_photos_qty",
        F.col("product_photos_qty").cast("int")
    )

    # Converte peso e dimensões para decimal
    .withColumn(
        "product_weight_g",
        F.col("product_weight_g").cast("decimal(18,2)")
    )

    .withColumn(
        "product_length_cm",
        F.col("product_length_cm").cast("decimal(18,2)")
    )

    .withColumn(
        "product_height_cm",
        F.col("product_height_cm").cast("decimal(18,2)")
    )

    .withColumn(
        "product_width_cm",
        F.col("product_width_cm").cast("decimal(18,2)")
    )
)

# Exibe uma amostra dos produtos tratados
display(df_products_silver.limit(10))

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,_data_ingestao,_arquivo_origem
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225.00,16.00,10.00,14.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000.00,30.00,18.00,20.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154.00,18.00,9.00,15.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371.00,26.00,4.00,26.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625.00,20.00,17.00,13.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200.00,38.00,5.00,11.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350.00,70.00,24.00,44.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900.00,40.00,8.00,40.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400.00,27.00,13.00,17.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600.00,17.00,10.00,12.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv


In [0]:
# Salva os produtos tratados na camada Silver
(
    df_products_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.products"
    )
)

print("Tabela products criada na Silver!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.products"
    ).count()
)

Tabela products criada na Silver!
Quantidade de registros: 32951


In [0]:
# Carrega os produtos persistidos na Silver
df_products_validacao = spark.table(
    f"{catalogo}.{schema_destino}.products"
)

# Verifica a quantidade de produtos e o tratamento das categorias
display(
    df_products_validacao.agg(

        F.count("*").alias("total_produtos"),

        F.countDistinct(
            "product_id"
        ).alias("produtos_unicos"),

        F.sum(
            F.when(
                F.col("product_category_name") == "nao_informada",
                1
            ).otherwise(0)
        ).alias("categorias_nao_informadas"),

        F.sum(
            F.when(
                F.col("product_category_name").isNull(),
                1
            ).otherwise(0)
        ).alias("categorias_nulas"),

        F.countDistinct(
            "product_category_name"
        ).alias("categorias_distintas")

    )
)

total_produtos,produtos_unicos,categorias_nao_informadas,categorias_nulas,categorias_distintas
32951,32951,610,0,74


In [0]:
# Carrega a tabela original de tradução das categorias
df_categories = spark.table(
    f"{catalogo}.{schema_origem}.category_translation"
)

# Padroniza os nomes das categorias
df_categories_silver = (
    df_categories

    .withColumn(
        "product_category_name",
        F.lower(F.trim(F.col("product_category_name")))
    )

    .withColumn(
        "product_category_name_english",
        F.lower(F.trim(F.col("product_category_name_english")))
    )
)

# Exibe uma amostra dos dados tratados
display(df_categories_silver.limit(10))

product_category_name,product_category_name_english,_data_ingestao,_arquivo_origem
beleza_saude,health_beauty,2026-09-22T23:37:44.242Z,product_category_name_translation.csv
informatica_acessorios,computers_accessories,2026-09-22T23:37:44.242Z,product_category_name_translation.csv
automotivo,auto,2026-09-22T23:37:44.242Z,product_category_name_translation.csv
cama_mesa_banho,bed_bath_table,2026-09-22T23:37:44.242Z,product_category_name_translation.csv
moveis_decoracao,furniture_decor,2026-09-22T23:37:44.242Z,product_category_name_translation.csv
esporte_lazer,sports_leisure,2026-09-22T23:37:44.242Z,product_category_name_translation.csv
perfumaria,perfumery,2026-09-22T23:37:44.242Z,product_category_name_translation.csv
utilidades_domesticas,housewares,2026-09-22T23:37:44.242Z,product_category_name_translation.csv
telefonia,telephony,2026-09-22T23:37:44.242Z,product_category_name_translation.csv
relogios_presentes,watches_gifts,2026-09-22T23:37:44.242Z,product_category_name_translation.csv


In [0]:
# Salva a tabela de tradução na camada Silver
(
    df_categories_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.category_translation"
    )
)

print("Tabela category_translation criada na Silver!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.category_translation"
    ).count()
)

Tabela category_translation criada na Silver!
Quantidade de registros: 71


In [0]:
# Carrega as tabelas Silver
df_products = spark.table(
    f"{catalogo}.{schema_destino}.products"
)

df_categories = spark.table(
    f"{catalogo}.{schema_destino}.category_translation"
)

# Enriquece os produtos com a tradução das categorias
df_products_enriched = (
    df_products.alias("p")

    .join(
        df_categories.alias("c"),

        F.col("p.product_category_name") ==
        F.col("c.product_category_name"),

        "left"
    )

    .select(
        "p.*",

        F.coalesce(
            F.col("c.product_category_name_english"),
            F.col("p.product_category_name")
        ).alias("categoria_analitica")
    )
)

# Exibe uma amostra dos produtos enriquecidos
display(df_products_enriched.limit(10))

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,_data_ingestao,_arquivo_origem,categoria_analitica
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225.00,16.00,10.00,14.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv,perfumery
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000.00,30.00,18.00,20.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv,art
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154.00,18.00,9.00,15.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv,sports_leisure
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371.00,26.00,4.00,26.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv,baby
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625.00,20.00,17.00,13.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv,housewares
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200.00,38.00,5.00,11.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv,musical_instruments
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350.00,70.00,24.00,44.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv,cool_stuff
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900.00,40.00,8.00,40.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv,furniture_decor
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400.00,27.00,13.00,17.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv,home_appliances
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600.00,17.00,10.00,12.00,2026-09-22T23:37:33.708Z,olist_products_dataset.csv,toys


In [0]:
# Atualiza a tabela de produtos com a categoria analítica
(
    df_products_enriched.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.products"
    )
)

print("Tabela products atualizada com categorias traduzidas!")

Tabela products atualizada com categorias traduzidas!


In [0]:
# Carrega os produtos enriquecidos
df_products_validacao = spark.table(
    f"{catalogo}.{schema_destino}.products"
)

# Verifica se os produtos foram preservados
display(
    df_products_validacao.agg(

        F.count("*").alias("total_produtos"),

        F.countDistinct(
            "product_id"
        ).alias("produtos_unicos"),

        F.sum(
            F.when(
                F.col("categoria_analitica") == "nao_informada",
                1
            ).otherwise(0)
        ).alias("categorias_nao_informadas"),

        F.sum(
            F.when(
                F.col("categoria_analitica").isNull(),
                1
            ).otherwise(0)
        ).alias("categorias_analiticas_nulas")

    )
)

total_produtos,produtos_unicos,categorias_nao_informadas,categorias_analiticas_nulas
32951,32951,610,0


In [0]:
# Carrega os pagamentos originais da camada Bronze
df_payments = spark.table(
    f"{catalogo}.{schema_origem}.order_payments"
)

# Padroniza os tipos e as informações de pagamento
df_payments_silver = (
    df_payments

    # Converte a sequência do pagamento para inteiro
    .withColumn(
        "payment_sequential",
        F.col("payment_sequential").cast("int")
    )

    # Converte a quantidade de parcelas para inteiro
    .withColumn(
        "payment_installments",
        F.col("payment_installments").cast("int")
    )

    # Padroniza a forma de pagamento
    .withColumn(
        "payment_type",
        F.lower(F.trim(F.col("payment_type")))
    )

    # Converte o valor financeiro para decimal
    .withColumn(
        "payment_value",
        F.col("payment_value").cast("decimal(18,2)")
    )
)

# Exibe uma amostra dos pagamentos tratados
display(df_payments_silver.limit(10))

order_id,payment_sequential,payment_type,payment_installments,payment_value,_data_ingestao,_arquivo_origem
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,2026-09-22T23:37:14.878Z,olist_order_payments_dataset.csv
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,2026-09-22T23:37:14.878Z,olist_order_payments_dataset.csv
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,2026-09-22T23:37:14.878Z,olist_order_payments_dataset.csv
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,2026-09-22T23:37:14.878Z,olist_order_payments_dataset.csv
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,2026-09-22T23:37:14.878Z,olist_order_payments_dataset.csv
298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12,2026-09-22T23:37:14.878Z,olist_order_payments_dataset.csv
771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16,2026-09-22T23:37:14.878Z,olist_order_payments_dataset.csv
3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84,2026-09-22T23:37:14.878Z,olist_order_payments_dataset.csv
1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09,2026-09-22T23:37:14.878Z,olist_order_payments_dataset.csv
0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95,2026-09-22T23:37:14.878Z,olist_order_payments_dataset.csv


In [0]:
# Salva os pagamentos tratados na camada Silver
(
    df_payments_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.order_payments"
    )
)

print("Tabela order_payments criada na Silver!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.order_payments"
    ).count()
)

Tabela order_payments criada na Silver!
Quantidade de registros: 103886


In [0]:
# Carrega os pagamentos persistidos na Silver
df_payments_validacao = spark.table(
    f"{catalogo}.{schema_destino}.order_payments"
)

# Verifica a quantidade de registros e os valores financeiros
display(
    df_payments_validacao.agg(

        F.count("*").alias("total_pagamentos"),

        F.countDistinct(
            "order_id"
        ).alias("pedidos_com_pagamento"),

        F.sum(
            "payment_value"
        ).alias("valor_total_pagamentos"),

        F.min(
            "payment_value"
        ).alias("menor_pagamento"),

        F.max(
            "payment_value"
        ).alias("maior_pagamento"),

        F.max(
            "payment_installments"
        ).alias("maior_parcelamento")

    )
)

total_pagamentos,pedidos_com_pagamento,valor_total_pagamentos,menor_pagamento,maior_pagamento,maior_parcelamento
103886,99440,16008872.12,0.00,13664.08,24


In [0]:
# Identifica as formas de pagamento utilizadas na Olist
display(
    df_payments_validacao

    .groupBy("payment_type")

    .agg(
        F.count("*").alias("quantidade_pagamentos"),

        F.sum("payment_value").alias("valor_total")
    )

    .orderBy(
        F.desc("quantidade_pagamentos")
    )
)

payment_type,quantidade_pagamentos,valor_total
credit_card,76795,12542084.19
boleto,19784,2869361.27
voucher,5775,379436.87
debit_card,1529,217989.79
not_defined,3,0.00


In [0]:
# Carrega as avaliações originais da camada Bronze
df_reviews = spark.table(
    f"{catalogo}.{schema_origem}.order_reviews"
)

# Padroniza os tipos e os campos textuais das avaliações
df_reviews_silver = (
    df_reviews

    # Converte a nota de avaliação para inteiro
    .withColumn(
        "review_score",
        F.col("review_score").cast("int")
    )

    # Converte as datas para timestamp
    .withColumn(
        "review_creation_date",
        F.to_timestamp("review_creation_date")
    )

    .withColumn(
        "review_answer_timestamp",
        F.to_timestamp("review_answer_timestamp")
    )

    # Remove espaços desnecessários dos comentários
    .withColumn(
        "review_comment_title",
        F.trim(F.col("review_comment_title"))
    )

    .withColumn(
        "review_comment_message",
        F.trim(F.col("review_comment_message"))
    )

    # Identifica avaliações que possuem comentário textual
    .withColumn(
        "possui_comentario",

        F.when(
            F.col("review_comment_message").isNotNull() &
            (F.col("review_comment_message") != ""),
            True
        ).otherwise(False)
    )
)

# Exibe uma amostra dos dados tratados
display(df_reviews_silver.limit(10))

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,_data_ingestao,_arquivo_origem,possui_comentario
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18T00:00:00.000Z,2018-01-18T21:46:59.000Z,2026-09-23T00:06:39.612Z,olist_order_reviews_dataset.csv,false
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10T00:00:00.000Z,2018-03-11T03:05:13.000Z,2026-09-23T00:06:39.612Z,olist_order_reviews_dataset.csv,false
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17T00:00:00.000Z,2018-02-18T14:36:24.000Z,2026-09-23T00:06:39.612Z,olist_order_reviews_dataset.csv,false
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21T00:00:00.000Z,2017-04-21T22:02:06.000Z,2026-09-23T00:06:39.612Z,olist_order_reviews_dataset.csv,true
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01T00:00:00.000Z,2018-03-02T10:26:53.000Z,2026-09-23T00:06:39.612Z,olist_order_reviews_dataset.csv,true
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13T00:00:00.000Z,2018-04-16T00:39:37.000Z,2026-09-23T00:06:39.612Z,olist_order_reviews_dataset.csv,false
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16T00:00:00.000Z,2017-07-18T19:30:34.000Z,2026-09-23T00:06:39.612Z,olist_order_reviews_dataset.csv,false
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14T00:00:00.000Z,2018-08-14T21:36:06.000Z,2026-09-23T00:06:39.612Z,olist_order_reviews_dataset.csv,false
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17T00:00:00.000Z,2017-05-18T12:05:37.000Z,2026-09-23T00:06:39.612Z,olist_order_reviews_dataset.csv,false
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22T00:00:00.000Z,2018-05-23T16:45:47.000Z,2026-09-23T00:06:39.612Z,olist_order_reviews_dataset.csv,true


In [0]:
# Salva as avaliações tratadas na camada Silver
(
    df_reviews_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.order_reviews"
    )
)

print("Tabela order_reviews criada na Silver!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.order_reviews"
    ).count()
)

Tabela order_reviews criada na Silver!
Quantidade de registros: 99224


In [0]:
# Carrega as avaliações persistidas na Silver
df_reviews_validacao = spark.table(
    f"{catalogo}.{schema_destino}.order_reviews"
)

# Valida os registros e os indicadores de satisfação
display(
    df_reviews_validacao.agg(

        F.count("*").alias("total_avaliacoes"),

        F.countDistinct(
            "order_id"
        ).alias("pedidos_avaliados"),

        F.round(
            F.avg("review_score"), 2
        ).alias("nota_media"),

        F.min(
            "review_score"
        ).alias("menor_nota"),

        F.max(
            "review_score"
        ).alias("maior_nota"),

        F.sum(
            F.when(
                F.col("possui_comentario") == True,
                1
            ).otherwise(0)
        ).alias("avaliacoes_com_comentario")

    )
)

total_avaliacoes,pedidos_avaliados,nota_media,menor_nota,maior_nota,avaliacoes_com_comentario
99224,98673,4.09,1,5,40968


In [0]:
# Calcula a distribuição das notas de avaliação
display(
    df_reviews_validacao

    .groupBy("review_score")

    .agg(
        F.count("*").alias("quantidade_avaliacoes")
    )

    .orderBy("review_score")
)

review_score,quantidade_avaliacoes
1,11424
2,3151
3,8179
4,19142
5,57328


In [0]:
# Carrega os vendedores originais da camada Bronze
df_sellers = spark.table(
    f"{catalogo}.{schema_origem}.sellers"
)

# Padroniza as informações geográficas dos vendedores
df_sellers_silver = (
    df_sellers

    # Padroniza o identificador do vendedor
    .withColumn(
        "seller_id",
        F.trim(F.col("seller_id"))
    )

    # Padroniza o CEP como texto de 5 dígitos
    .withColumn(
        "seller_zip_code_prefix",
        F.lpad(
            F.trim(F.col("seller_zip_code_prefix")),
            5,
            "0"
        )
    )

    # Padroniza os nomes das cidades
    .withColumn(
        "seller_city",
        F.lower(F.trim(F.col("seller_city")))
    )

    # Padroniza as siglas dos estados
    .withColumn(
        "seller_state",
        F.upper(F.trim(F.col("seller_state")))
    )
)

# Exibe uma amostra dos vendedores tratados
display(df_sellers_silver.limit(10))

seller_id,seller_zip_code_prefix,seller_city,seller_state,_data_ingestao,_arquivo_origem
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,2026-09-22T23:37:39.507Z,olist_sellers_dataset.csv
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,2026-09-22T23:37:39.507Z,olist_sellers_dataset.csv
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,2026-09-22T23:37:39.507Z,olist_sellers_dataset.csv
c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP,2026-09-22T23:37:39.507Z,olist_sellers_dataset.csv
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,2026-09-22T23:37:39.507Z,olist_sellers_dataset.csv
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ,2026-09-22T23:37:39.507Z,olist_sellers_dataset.csv
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE,2026-09-22T23:37:39.507Z,olist_sellers_dataset.csv
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP,2026-09-22T23:37:39.507Z,olist_sellers_dataset.csv
768a86e36ad6aae3d03ee3c6433d61df,01529,sao paulo,SP,2026-09-22T23:37:39.507Z,olist_sellers_dataset.csv
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR,2026-09-22T23:37:39.507Z,olist_sellers_dataset.csv


In [0]:
# Salva os vendedores tratados na camada Silver
(
    df_sellers_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.sellers"
    )
)

print("Tabela sellers criada na Silver!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.sellers"
    ).count()
)

Tabela sellers criada na Silver!
Quantidade de registros: 3095


In [0]:
# Carrega os vendedores persistidos na Silver
df_sellers_validacao = spark.table(
    f"{catalogo}.{schema_destino}.sellers"
)

# Verifica a integridade e a distribuição geográfica
display(
    df_sellers_validacao.agg(

        F.count("*").alias("total_vendedores"),

        F.countDistinct(
            "seller_id"
        ).alias("vendedores_unicos"),

        F.countDistinct(
            "seller_state"
        ).alias("estados_distintos"),

        F.countDistinct(
            "seller_city"
        ).alias("cidades_distintas"),

        F.sum(
            F.when(
                F.col("seller_state").isNull(),
                1
            ).otherwise(0)
        ).alias("estados_nulos"),

        F.sum(
            F.when(
                F.length("seller_zip_code_prefix") != 5,
                1
            ).otherwise(0)
        ).alias("ceps_invalidos")

    )
)

total_vendedores,vendedores_unicos,estados_distintos,cidades_distintas,estados_nulos,ceps_invalidos
3095,3095,23,611,0,0


In [0]:
# Identifica a concentração geográfica dos vendedores
display(
    df_sellers_validacao

    .groupBy("seller_state")

    .agg(
        F.countDistinct("seller_id").alias(
            "quantidade_vendedores"
        )
    )

    .orderBy(
        F.desc("quantidade_vendedores")
    )
)

seller_state,quantidade_vendedores
SP,1849
PR,349
MG,244
SC,190
RJ,171
RS,129
GO,40
DF,30
ES,23
BA,19


In [0]:
# Carrega os dados geográficos originais da Bronze
df_geo = spark.table(
    f"{catalogo}.{schema_origem}.geolocation"
)

# Padroniza os atributos geográficos
df_geo_silver = (
    df_geo

    # Padroniza o prefixo do CEP para cinco caracteres
    .withColumn(
        "geolocation_zip_code_prefix",
        F.lpad(
            F.trim(F.col("geolocation_zip_code_prefix")),
            5,
            "0"
        )
    )

    # Converte latitude para decimal
    .withColumn(
        "geolocation_lat",
        F.col("geolocation_lat").cast("double")
    )

    # Converte longitude para decimal
    .withColumn(
        "geolocation_lng",
        F.col("geolocation_lng").cast("double")
    )

    # Padroniza o nome da cidade
    .withColumn(
        "geolocation_city",
        F.lower(F.trim(F.col("geolocation_city")))
    )

    # Padroniza a sigla do estado
    .withColumn(
        "geolocation_state",
        F.upper(F.trim(F.col("geolocation_state")))
    )

    # Remove registros completamente duplicados
    .dropDuplicates()
)

# Exibe uma amostra dos dados tratados
display(df_geo_silver.limit(10))

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,_data_ingestao,_arquivo_origem
01037,-23.54562128115268,-46.63929204800168,sao paulo,SP,2026-09-22T23:36:59.844Z,olist_geolocation_dataset.csv
01046,-23.546081127035535,-46.64482029837157,sao paulo,SP,2026-09-22T23:36:59.844Z,olist_geolocation_dataset.csv
01046,-23.54612896641469,-46.64295148361138,sao paulo,SP,2026-09-22T23:36:59.844Z,olist_geolocation_dataset.csv
01041,-23.5443921648681,-46.63949930627844,sao paulo,SP,2026-09-22T23:36:59.844Z,olist_geolocation_dataset.csv
01035,-23.541577961711493,-46.64160722329613,sao paulo,SP,2026-09-22T23:36:59.844Z,olist_geolocation_dataset.csv
01012,-23.547762303364266,-46.63536053788448,são paulo,SP,2026-09-22T23:36:59.844Z,olist_geolocation_dataset.csv
01047,-23.546273112412678,-46.64122516971552,sao paulo,SP,2026-09-22T23:36:59.844Z,olist_geolocation_dataset.csv
01013,-23.546923208436723,-46.6342636964915,sao paulo,SP,2026-09-22T23:36:59.844Z,olist_geolocation_dataset.csv
01029,-23.543769055769133,-46.63427784085132,sao paulo,SP,2026-09-22T23:36:59.844Z,olist_geolocation_dataset.csv
01011,-23.547639550320632,-46.63603162315495,sao paulo,SP,2026-09-22T23:36:59.844Z,olist_geolocation_dataset.csv


In [0]:
# Salva os dados geográficos tratados na Silver
(
    df_geo_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalogo}.{schema_destino}.geolocation"
    )
)

print("Tabela geolocation criada na Silver!")

print(
    "Quantidade de registros:",
    spark.table(
        f"{catalogo}.{schema_destino}.geolocation"
    ).count()
)

Tabela geolocation criada na Silver!
Quantidade de registros: 738332


In [0]:
# Carrega os dados geográficos persistidos na Silver
df_geo_validacao = spark.table(
    f"{catalogo}.{schema_destino}.geolocation"
)

# Verifica a integridade dos dados geográficos
display(
    df_geo_validacao.agg(

        F.count("*").alias("total_registros"),

        F.countDistinct(
            "geolocation_zip_code_prefix"
        ).alias("ceps_distintos"),

        F.countDistinct(
            "geolocation_state"
        ).alias("estados_distintos"),

        F.sum(
            F.when(
                F.col("geolocation_lat").isNull() |
                F.col("geolocation_lng").isNull(),
                1
            ).otherwise(0)
        ).alias("coordenadas_nulas"),

        F.sum(
            F.when(
                (F.col("geolocation_lat") < -90) |
                (F.col("geolocation_lat") > 90) |
                (F.col("geolocation_lng") < -180) |
                (F.col("geolocation_lng") > 180),
                1
            ).otherwise(0)
        ).alias("coordenadas_invalidas"),

        F.sum(
            F.when(
                F.length("geolocation_zip_code_prefix") != 5,
                1
            ).otherwise(0)
        ).alias("ceps_invalidos")

    )
)

total_registros,ceps_distintos,estados_distintos,coordenadas_nulas,coordenadas_invalidas,ceps_invalidos
738332,19015,27,0,0,0


In [0]:
# Compara a quantidade de registros entre Bronze e Silver
total_bronze = spark.table(
    f"{catalogo}.{schema_origem}.geolocation"
).count()

total_silver = spark.table(
    f"{catalogo}.{schema_destino}.geolocation"
).count()

print("Registros na Bronze:", total_bronze)

print("Registros na Silver:", total_silver)

print(
    "Duplicatas completas removidas:",
    total_bronze - total_silver
)

Registros na Bronze: 1000163
Registros na Silver: 738332
Duplicatas completas removidas: 261831
